# Atualização para a v2: controles tóxicos, RQ5 e relação da contribuição D com a PSA

Três análises das contribuições C e D foram feitas com a v1.1, antes da v2 existir. Este notebook as refaz com a v2, que é a base das análises principais do artigo:

1. **Controles tóxicos por categoria emocional:** a v2 tem 145 controles (130 na v1.1), com as palavras emocionais corrigidas.
2. **RQ5, termo nos dados de treino × efeito na PSA:** passa a usar os efeitos da v2, incluindo os quatro termos novos (autista, deficiente, deficiente mental e em cadeira de rodas).
3. **Relação da contribuição D com a PSA:** compara as taxas de falso positivo do ToxSyn-PT e do HateCheck-PT com os termos correspondentes na v2.

**Entrada:** `data_bits/BITS_PTBR_v2_corrigido_com_predicoes.csv`, `lista_termos_congelada.csv` e, em `results`, `thresholds.json`, `v2_bootstrap_contexto_termo.csv`, `c1_contagem_termos_por_corpus.csv`, `c1_inspecao_corpora.csv`, `d_fpr_toxsyn_por_grupo.csv` e `d_fpr_hatecheck_por_grupo.csv`.

**Saída (em `results`):** `c5_v2_controles_toxicos_por_categoria.csv`, `c6_v2_rq5_termos_treino_x_efeito.csv` e `d_v2_relacao_psa.csv`. Os arquivos da v1.1 (`c5_*`, `c6_*`) não são sobrescritos.

**Não precisa de GPU:** usa os scores já calculados.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import ast, json
import numpy as np, pandas as pd

ROOT = "/content/drive/MyDrive/JIS - capacitismo"
DATA = f"{ROOT}/data_bits"
OUT  = f"{ROOT}/results"
ORDEM = ["padrao_0.5", "youden", "fpr_5"]
MODELOS = {"tox1": "BERTimbau toxic", "tox2": "ToxiGuardrail-PT"}
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)

with open(f"{OUT}/thresholds.json") as f:
    thresholds = json.load(f)
v2 = pd.read_csv(f"{DATA}/BITS_PTBR_v2_corrigido_com_predicoes.csv", encoding="utf-8-sig")
termos = pd.read_csv(f"{ROOT}/lista_termos_congelada.csv")
boot = pd.read_csv(f"{OUT}/v2_bootstrap_contexto_termo.csv")
assert len(v2) == 5863
print("v2:", len(v2), "pares |", v2.Sentence_control.nunique(), "controles | limiares:", thresholds)

## 1. Controles tóxicos por categoria emocional (v2)

Para cada limiar, a fração dos 145 controles distintos que o modelo marca como tóxicos, por categoria emocional e por template. Os controles não contêm nenhum termo, então essa taxa é o ponto de partida da PSA.

In [ ]:
ctrl = v2.drop_duplicates("Sentence_control")[["Template", "emocao_categoria", "Sentence_control",
                                               "tox1_score_control", "tox2_score_control"]].reset_index(drop=True)
assert len(ctrl) == 145

rows = []
for key, modelo in MODELOS.items():
    for lim in ORDEM:
        toxico = ctrl[f"{key}_score_control"] >= thresholds[key][lim]
        for recorte in ["emocao_categoria", "Template"]:
            for valor, g in toxico.groupby(ctrl[recorte]):
                rows.append({"modelo": modelo, "limiar": lim, "recorte": recorte, "valor": valor,
                             "controles": len(g), "toxicos": int(g.sum()), "taxa": g.mean()})
        rows.append({"modelo": modelo, "limiar": lim, "recorte": "total", "valor": "todos",
                     "controles": len(toxico), "toxicos": int(toxico.sum()), "taxa": toxico.mean()})
c5 = pd.DataFrame(rows)
c5.to_csv(f"{OUT}/c5_v2_controles_toxicos_por_categoria.csv", index=False)

for modelo in MODELOS.values():
    t = c5[c5.modelo == modelo]
    print(f"\n{modelo}, total:")
    print(t[t.recorte == "total"].set_index("limiar").loc[ORDEM, ["controles", "toxicos", "taxa"]].round(3))
    print(f"{modelo}, por categoria emocional (taxa):")
    print(t[t.recorte == "emocao_categoria"].pivot(index="valor", columns="limiar", values="taxa")[ORDEM]
          .sort_values("padrao_0.5", ascending=False).round(3))
t = c5[(c5.modelo == "BERTimbau toxic") & (c5.recorte == "Template")]
print("\nBERTimbau toxic, por template (taxa):")
print(t.pivot(index="valor", columns="limiar", values="taxa")[ORDEM].round(3).T.to_string())

## 2. RQ5: o termo nos dados de treino e o efeito dele na v2

Para cada termo de deficiência da lista congelada:
- **No treino:** quantos textos com o termo estão do lado nocivo e do inofensivo. Para o ToxiGuardrail-PT, o treino é o `harmful-text`; para o BERTimbau toxic, o OLID-BR. As contagens vêm do notebook de inspeção.
- **Proporção nociva:** só calculada quando o termo aparece em pelo menos 10 textos; deve ser comparada com a do corpus inteiro (50% no `harmful-text` e 85% no OLID-BR).
- **Efeito na v2:** ScoreSense com intervalo de 95% por contexto (`v2_bootstrap_contexto_termo.csv`) e fração de pares que passam de não tóxico a tóxico com o limiar de 5% de falsos positivos.

A comparação é descritiva: com 16 termos e dois modelos, não cabe calcular correlação.

In [ ]:
contagem = pd.read_csv(f"{OUT}/c1_contagem_termos_por_corpus.csv")
inspecao = pd.read_csv(f"{OUT}/c1_inspecao_corpora.csv")

def contar(corpus, nocivo, nao_nocivo):
    c = (contagem[contagem.corpus == corpus]
         .pivot_table(index="termo", columns="rotulo", values="textos", aggfunc="sum")
         .reindex(columns=[nocivo, nao_nocivo]).fillna(0).astype(int))
    c.columns = ["nocivo", "nao_nocivo"]
    return c

def taxa_base(corpus, nocivo):
    r = ast.literal_eval(inspecao.set_index("corpus").loc[corpus, "rotulos"])
    return r[nocivo] / sum(r.values())

MIN_TEXTOS = 10
TREINOS = {"harmful_text": ("harmful-text (PT)", "harmful", "harmless"),
           "olid":         ("OLID-BR", "ofensivo", "não ofensivo")}

alvo = termos[termos.grupo != "Só E (xingamento)"][["termo", "grupo"]].copy()
alvo["SubClass"] = [next((s for s in v2.SubClass.unique() if s.lower() == t.lower()), t) for t in alvo.termo]
assert alvo.SubClass.isin(v2.SubClass).all(), alvo[~alvo.SubClass.isin(v2.SubClass)]

for key, (corpus, nocivo, nao_nocivo) in TREINOS.items():
    c = contar(corpus, nocivo, nao_nocivo).reindex(alvo.termo).fillna(0).astype(int)
    total = c.nocivo + c.nao_nocivo
    alvo[f"{key}_nocivo"] = c.nocivo.values
    alvo[f"{key}_nao_nocivo"] = c.nao_nocivo.values
    alvo[f"{key}_prop_nocivo"] = np.where(total.values >= MIN_TEXTOS, (c.nocivo / total.replace(0, np.nan)).values, np.nan)
    alvo[f"{key}_prop_base"] = taxa_base(corpus, nocivo)

por_termo = boot[(boot.Nivel == "Por termo") & (boot.Bootstrap == "Contexto")]
for key, modelo, prefixo in [("tox2", "ToxiGuardrail-PT", "toxiguardrail"), ("tox1", "BERTimbau toxic", "bertimbau")]:
    b = por_termo[por_termo.Modelo == modelo].set_index("Termo")
    alvo[f"{prefixo}_score_sense"] = alvo.SubClass.map(b.ScoreSense)
    alvo[f"{prefixo}_ic95_inf"] = alvo.SubClass.map(b.IC95_inf)
    alvo[f"{prefixo}_ic95_sup"] = alvo.SubClass.map(b.IC95_sup)
    lim = thresholds[key]["fpr_5"]
    para_toxico = ((v2[f"{key}_score_control"] < lim) & (v2[f"{key}_score_group"] >= lim)).groupby(v2.SubClass).mean()
    alvo[f"{prefixo}_flip_para_toxico_fpr5"] = alvo.SubClass.map(para_toxico)

alvo.to_csv(f"{OUT}/c6_v2_rq5_termos_treino_x_efeito.csv", index=False)
print("Base: harmful-text", round(taxa_base("harmful-text (PT)", "harmful"), 3), "| OLID-BR", round(taxa_base("OLID-BR", "ofensivo"), 3))
print("\nToxiGuardrail-PT: termo no treino (harmful-text) × efeito na v2")
print(alvo[["termo", "harmful_text_nocivo", "harmful_text_nao_nocivo", "harmful_text_prop_nocivo",
            "toxiguardrail_score_sense", "toxiguardrail_ic95_inf", "toxiguardrail_ic95_sup",
            "toxiguardrail_flip_para_toxico_fpr5"]].round(3).to_string(index=False))
print("\nBERTimbau toxic: termo no treino (OLID-BR) × efeito na v2")
print(alvo[["termo", "olid_nocivo", "olid_nao_nocivo", "olid_prop_nocivo",
            "bertimbau_score_sense", "bertimbau_ic95_inf", "bertimbau_ic95_sup",
            "bertimbau_flip_para_toxico_fpr5"]].round(3).to_string(index=False))

## 3. Relação da contribuição D com a PSA (v2)

O ToxSyn-PT usa quase só a forma institucional "pessoas com deficiência", e o HateCheck-PT usa "deficientes". A célula põe lado a lado a taxa de falso positivo desses grupos e a fração de pares que passam a tóxicos na PSA da v2 para os termos correspondentes ("pessoa com deficiência" e "deficiente") e para "cego", "surdo" e "cadeirante".

In [ ]:
TERMOS_D = ["pessoa com deficiência", "deficiente", "cego", "surdo", "cadeirante"]
linhas = []
for key, modelo in MODELOS.items():
    for lim in ORDEM:
        v = thresholds[key][lim]
        for termo in TERMOS_D:
            d = v2[v2.SubClass == termo]
            linhas.append({"modelo": modelo, "limiar": lim, "fonte": "PSA v2 (pares que passam a tóxico)", "item": termo,
                           "valor": ((d[f"{key}_score_control"] < v) & (d[f"{key}_score_group"] >= v)).mean()})
ts = pd.read_csv(f"{OUT}/d_fpr_toxsyn_por_grupo.csv")
hc = pd.read_csv(f"{OUT}/d_fpr_hatecheck_por_grupo.csv")
for df, fonte, grupo, item in [(ts, "ToxSyn-PT (taxa de FP)", "pessoas com deficiência", "\"pessoas com deficiência\""),
                               (hc, "HateCheck-PT (taxa de FP)", "disabled people", "\"deficientes\"")]:
    d = df[df.grupo == grupo]
    for _, r in d.iterrows():
        linhas.append({"modelo": r.modelo, "limiar": r.limiar, "fonte": fonte, "item": item, "valor": r.taxa_fp})
rel = pd.DataFrame(linhas)
rel.to_csv(f"{OUT}/d_v2_relacao_psa.csv", index=False)
print(rel.pivot_table(index=["modelo", "fonte", "item"], columns="limiar", values="valor", sort=False)[ORDEM].round(3).to_string())

## Leitura dos resultados

**1. Controles tóxicos (v2).** No BERTimbau toxic, 48 dos 145 controles (33,1%) saem como tóxicos com o limiar de 0,5; 12 (8,3%) com o de Youden; e 5 (3,4%) com o de 5% de falsos positivos. No ToxiGuardrail-PT, nenhum, em nenhum limiar. Com 0,5, a taxa é de 93% em nojo e 87% em raiva, de 33% a 47% em medo, tristeza e surpresa negativa, e de 4% a 7% em felicidade, surpresa positiva e neutro; nenhum controle dos 15 templates sem emoção sai como tóxico. O padrão da v1.1 se mantém: o score de base vem da palavra emocional. O T10 ("A pessoa estava em uma situação ...") continua sendo o template com a maior taxa (50% com 0,5), agora sem o erro de concordância da v1, o que indica que a taxa alta vem da palavra "situação" combinada a eventos negativos, e não só da concordância.

**2. RQ5 (v2).**
- *ToxiGuardrail-PT:* os termos que aparecem quase só do lado nocivo do treino têm efeito claro: "cego" (98% nocivo; ScoreSense 0,141), "surdo" (99,5%; 0,174), "deficiente" (99%; 0,162) e "deficiente mental" (100%; 0,246). "Pessoa com deficiência", que aparece dos dois lados (56%), tem efeito pequeno (0,020). A previsão feita com a v1.1 para "deficiente" e "deficiente mental" se confirmou. Mas a associação continua parcial: "cadeirante" não aparece no treino e tem efeito de 0,212, e "autista", com 4 textos, tem 0,249. Na decisão com o limiar de 5% de falsos positivos, os termos que mais mudam são "autista" e "deficiente mental" (19%) e "cadeirante" (15%), enquanto "cego" quase não muda (1,4%).
- *BERTimbau toxic:* os termos de deficiência são raros no OLID-BR (no máximo 19 textos, de "cego", 74% ofensivos, abaixo dos 85% do corpus), e mesmo assim os efeitos são grandes: "deficiente mental" 0,514 (83% dos pares passam a tóxicos com o limiar de 5% de FP), "cego" 0,450 (57%), "surdo" 0,395 (35%) e "deficiente" 0,334 (33%). Os dados de ajuste do modelo não explicam o efeito; ele provavelmente vem do pré-treino ou de associações mais gerais, hipótese não testada aqui.

**3. Relação com a PSA (v2).** A forma de nomear explica a diferença entre os dois corpora externos também na v2. "Pessoa com deficiência" não muda a decisão na PSA (0% dos pares passam a tóxicos com o limiar de 5% de FP nos dois modelos), assim como "pessoas com deficiência" no ToxSyn-PT (taxa de falso positivo de 0%). "Deficiente" muda a decisão em 33% dos pares no BERTimbau toxic, valor próximo da taxa de falso positivo de "deficientes" no HateCheck-PT (37%). No ToxiGuardrail-PT, "deficiente" muda 2,8% dos pares, e "deficientes" no HateCheck-PT têm 13,7% de falsos positivos; os dois são baixos, mas a convergência é menor.